# Binary Baseline Replication

Reproduce the eight per-emotion binary baselines from Sosea & Caragea (2020) Table 4 on Dataset A.

Wraps `scripts/binary_baseline_replication.py`, which runs two internal phases:

1. Classical Tier-1 (Majority, TF-IDF + LR, TF-IDF + NB) — CPU, < 1 min.
2. Transformer (BERT or DistilBERT) — GPU recommended; targets the original BERT macro F1 ≈ 0.71.

Use `--skip-bert` for CPU-only runs.

## 0. Install / check packages

In [ ]:
import importlib.util
for pkg in ('numpy', 'pandas', 'sklearn', 'torch', 'transformers'):
    print(f"{pkg:14s}: {'OK' if importlib.util.find_spec(pkg) else 'MISSING'}")

## 1. Setup

In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.resolve()
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)

## 2. Phase 1 — classical Tier-1 baselines (CPU)

In [ ]:
!python scripts/binary_baseline_replication.py --skip-bert

## 3. Per-emotion classical F1

In [ ]:
import pandas as pd
out = REPO_ROOT / '02_BinaryBaselineReplication' / 'outputs'
metrics = pd.read_csv(out / '02_baseline_metrics.csv')
metrics.pivot(index='emotion', columns='model', values='f1').round(3)

## 4. Classical macro averages

In [ ]:
pd.read_csv(out / '02_macro_average_f1.csv').round(3)

## 5. Phase 2 — transformer baseline (GPU recommended)

On Colab T4 / A100. For a CPU smoke-test, replace `bert-base-uncased` with `distilbert-base-uncased` and consider `--emotions Anticipation` to scope down.

In [ ]:
!python scripts/binary_baseline_replication.py \
    --skip-classical \
    --bert-model bert-base-uncased \
    --epochs 3 --batch-size 32 --max-length 128 --lr 2e-5 --seed 42

## 6. Per-emotion transformer metrics

In [ ]:
pd.read_csv(out / '02_bert_metrics.csv').round(3)